# Figure 6 — Per-Action Physical Direction and Per-Goal Policy Structure

For the GA-optimised wrap_grid twist, show:

1. **Physical direction per label** with stationary distribution labels —
   quiver arrows showing the actual transition at each state, with $d^a(s)$
   values annotated.
2. **Dominant action label per state** for all 49 goals in a 7×7 grid —
   each panel is positioned at the goal's grid location.
3. **Decision information per state** for all 49 goals in the same 7×7 grid.

Run reference locked in `figure-6-refs.json`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# gridvis pilot: import from installed packages (gridcore / gridvis / gridbench).
# No gridFour on sys.path. The support module + figures are co-located here.
NOTEBOOK_ROOT = Path.cwd()
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from gridcore.envs import GridRoom
from figure_support import build_env, save_figure_variants
from gridvis import display
from gridvis.display_twist import _action_labels, _action_color_indices, _select_action_colours
from gridvis.display_twist_analysis import (
    plot_physical_direction_row,
    plot_stationary_row,
    plot_dominant_action_panel,
    plot_stationary_panel,
    plot_goal_grid,
    apply_dashed_border,
    apply_attractor_panel_border,
    SOFT_STATIONARY,
)
from gridcore.planning.state_distribution import get_stationary_distribution_eigen_decomposition

mpl.rcParams['mathtext.fontset'] = 'cm'

FIGURES_DIR = NOTEBOOK_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

## Load config and sigma

In [ ]:
from figure_support import WRAP_GRID_RUN_DIR, load_run_config

cfg = load_run_config(WRAP_GRID_RUN_DIR)

RUN_DIR = cfg['run_dir']
RUN_NAME = cfg['run_name']
SIGMA_HASH = cfg['best_sigma_hash']
SIGMA_PATH = cfg['sigma_path']
HIVE_DIR = cfg['hive_dir']
FOOTER = f"$\\sigma$ = {SIGMA_HASH}    run = {RUN_NAME}"

sigma = np.load(SIGMA_PATH)

print(f'Run:       {RUN_NAME}')
print(f'Sigma:     {SIGMA_HASH}  shape={sigma.shape}')
print(f'chi_twist: {cfg["chi_twist"]:.4f}')
print(f'mean_free: {cfg["mean_free"]:.3f}')
print(f'mean_info: {cfg["mean_info"]:.3f}')
print(f'Hive:      {HIVE_DIR}')

In [ ]:
# build twisted environment — two variants:
#
# 1. `env` (no goal absorbing): used for per-action (per-label) dynamics
#    analysis in the TOP ROW. The per-label stationary distribution and
#    limit cycles must not depend on which goal is solved, so we build
#    with goals=[] to avoid any goal-state absorption contaminating the
#    deterministic skeleton.
#
# 2. `build_twisted_env(goal=g)`: goal-conditioned. Used for per-goal policy
#    plots in the BOTTOM ROW (dominant action, DI) — these are inherently
#    goal-specific and should have the goal state absorbing.

def build_twisted_env(goal: int = 0) -> GridRoom:
    """Goal-conditioned twisted env (for per-goal policy plots)."""
    env = build_env(cfg['env_id'], shape=tuple(cfg['shape']),
                    goal=goal, determinism=cfg['determinism'])
    available = np.asarray(env.available_states, dtype=int)
    # set_sigma is the only supported way to mutate sigma post-construction:
    # direct assignment leaves sigma_inv stale (identity), which silently
    # untwists the display_twist arrow helpers (2026-05-23 audit).
    sigma_full = np.tile(np.arange(env.nA, dtype=int), (env.nS, 1))
    sigma_full[available] = sigma[available]
    env.set_sigma(sigma_full)
    env.twist_dynamics()
    env.update_dynamics_for_goals(env.goals)
    return env


def build_twisted_env_no_goal() -> GridRoom:
    """Unconditioned twisted env (for per-action dynamics analysis).

    Builds a GridRoom with goals=[] so no state is absorbing. ``check_goals([])``
    is vacuously true; the for-loop in ``update_dynamics_for_goals`` over
    ``self.goals`` is empty, so the transition kernel has no goal-state
    modifications.
    """
    options = {
        'shape': tuple(cfg['shape']),
        'goals': [],
        'manhattan': True,
        'determinism': float(cfg['determinism']),
        'epsilon': 0.0,
        'twist_seed': 0,
        'wrap': (cfg['env_id'] == 'wrap_grid'),
    }
    env = GridRoom(options)
    available = np.asarray(env.available_states, dtype=int)
    # set_sigma is the only supported way to mutate sigma post-construction:
    # direct assignment leaves sigma_inv stale (identity), which silently
    # untwists the display_twist arrow helpers (2026-05-23 audit).
    sigma_full = np.tile(np.arange(env.nA, dtype=int), (env.nS, 1))
    sigma_full[available] = sigma[available]
    env.set_sigma(sigma_full)
    env.twist_dynamics()
    env.update_dynamics_for_goals(env.goals)  # goals=[] so no-op
    return env


# Environment for per-action dynamics (top row, attractor identification,
# stationary distribution): NO absorbing goal.
env = build_twisted_env_no_goal()
ACTION_LABELS = _action_labels(env)
h, w = env.shape
nS = env.nS
print(f'nS={nS}  shape={env.shape}  goals={list(env.goals)} (empty = unconditioned)')


## Panel A: Physical direction per label with occupancy labels

Quiver arrows showing the actual physical transition (through $\sigma$),
with $d^a(s)$ heatmap underlay and cell annotations.

In [ ]:
# per-action occupancy d^a(s) — cached by sigma hash + goal-condition tag
# so goal-conditioned and unconditioned variants don't collide.
CACHE_DIR = FIGURES_DIR / '_cache'
CACHE_DIR.mkdir(exist_ok=True)
goal_tag = 'nogoal' if not list(env.goals) else 'goal' + '_'.join(str(int(g)) for g in sorted(env.goals))
stationary_cache = CACHE_DIR / f'stationary-{SIGMA_HASH[:16]}-{goal_tag}.npz'

if stationary_cache.exists():
    cached = np.load(stationary_cache)
    stationary = {a: cached[f'action_{a}'] for a in range(env.nA)}
    print(f'Loaded stationary distributions from cache: {stationary_cache.name}')
else:
    stationary = {}
    for a in range(env.nA):
        pi = np.zeros((env.nS, env.nA))
        pi[:, a] = 1.0
        stationary[a] = get_stationary_distribution_eigen_decomposition(env, pi)
    np.savez_compressed(stationary_cache,
                        **{f'action_{a}': stationary[a] for a in range(env.nA)})
    print(f'Computed and cached stationary distributions: {stationary_cache.name}')

# physical direction row: quiver + heatmap + d^a labels (no attractor letters)
fig_dir, _ = plot_physical_direction_row(
    env, stationary,
    show_attractors=False,
    show_action_legend=True,
    label=True,
    footer_text=FOOTER,
)
save_figure_variants(fig_dir, FIGURES_DIR / 'figure-6-physical-direction')
plt.show()

# also the stationary distribution with labels for reference
fig_occ, _ = plot_stationary_row(
    env, stationary,
    footer_text=FOOTER,
)
save_figure_variants(fig_occ, FIGURES_DIR / 'figure-6-occupancy')
plt.show()


## Panel A2: Functional graph decomposition (label cycles and basins)

For each label, treat the noisy dynamics as a deterministic map by taking
the most-probable successor at every state (the `det = 1` skeleton). The
resulting functional graph decomposes into limit cycles plus tree-shaped
basins of attraction.

Under the GA-best twist on the wrap_grid, three of the four labels collapse
to a single short cycle covering the whole grid as one basin. **Label E is
the exception:** it partitions the grid into three disjoint basins, each
draining to its own 2-cycle. Two of those cycles are large (basin sizes 27
and 20), and the third is a tiny isolated 2-cycle (basin size 2) that gets
drained by noise once you switch the noise back on.

Cycle states are hatched in the colour of their label. Coloured regions
show the basin partition (Set2 colourmap).


In [ ]:
# Functional graph decomposition under repeated single-label dynamics.
# Uses the unconditioned env (`env`) so the per-label cycles are a property
# of the twist alone, not of any particular goal.

from matplotlib.patches import Rectangle as _Rect
from gridvis.display_twist import _action_color_indices, _select_action_colours

# Local imports of the basin/successor helpers (defined in wrap_grid_helpers
# but the wrap_grid notebook folder isn't on this notebook's path).
def _deterministic_successor(env, action):
    T = np.asarray(env.T)
    succ = np.zeros(env.nS, dtype=int)
    for s in range(env.nS):
        row = T[s * env.nA + action]
        succ[s] = int(np.argmax(row))
    return succ


def _find_basins(succ):
    n = len(succ)
    visited = np.full(n, -1, dtype=int)
    basin_id = np.full(n, -1, dtype=int)
    cycles = []
    current_basin = 0

    for start in range(n):
        if visited[start] >= 0:
            continue
        path = []
        s = start
        while visited[s] < 0:
            visited[s] = start
            path.append(s)
            s = succ[s]

        if visited[s] == start:
            cycle_start_idx = path.index(s)
            cycle = path[cycle_start_idx:]
            cycles.append(cycle)
            bid = current_basin
            current_basin += 1
            for c in cycle:
                basin_id[c] = bid
            for c in path[:cycle_start_idx]:
                basin_id[c] = bid
        else:
            bid = basin_id[s]
            for c in path:
                basin_id[c] = bid

    basin_sizes = [int(np.sum(basin_id == b)) for b in range(current_basin)]
    return {'cycles': cycles, 'basin_id': basin_id, 'basin_sizes': basin_sizes}


# Action colours for hatching
_cids_d, _ps_d = _action_color_indices(ACTION_LABELS)
_action_colors_d = _select_action_colours(_ps_d)

fig_decomp, axes_decomp = plt.subplots(1, env.nA, figsize=(4 * env.nA + 0.5, 4.2))
basin_cmap = plt.cm.Set2

for ax_idx, a in enumerate(range(env.nA)):
    ax = axes_decomp[ax_idx]
    succ = _deterministic_successor(env, a)
    result = _find_basins(succ)
    n_basins = len(result['cycles'])

    # Basin colours
    grid = result['basin_id'].reshape(env.shape).astype(float)
    ax.imshow(grid, cmap=basin_cmap, vmin=-0.5, vmax=max(n_basins - 0.5, 0.5),
              origin='upper', interpolation='nearest')

    # Cycle states for hatching
    cycle_states = set()
    for cycle in result['cycles']:
        for s in cycle:
            cycle_states.add(int(s))

    label_colour = _action_colors_d[_cids_d[a]]
    h_grid, w_grid = env.shape
    for r in range(h_grid):
        for c in range(w_grid):
            s = r * w_grid + c
            s_next = succ[s]
            r2, c2 = divmod(int(s_next), w_grid)
            dr = r2 - r
            dc = c2 - c
            if abs(dr) > 1:
                dr = -np.sign(dr)
            if abs(dc) > 1:
                dc = -np.sign(dc)

            # Hatch cycle cells in label colour
            if s in cycle_states:
                ax.add_patch(_Rect(
                    (c - 0.5, r - 0.5), 1.0, 1.0,
                    facecolor='none', edgecolor=label_colour,
                    linewidth=0.0, hatch='///', alpha=0.7, zorder=2,
                ))

            if dr == 0 and dc == 0:
                ax.plot(c, r, 'o', color='red', markersize=4, zorder=4)
            else:
                ax.annotate('', xy=(c + dc * 0.4, r + dr * 0.4),
                            xytext=(c - dc * 0.1, r - dr * 0.1),
                            arrowprops=dict(arrowstyle='->', color='black',
                                            lw=1.0, mutation_scale=8),
                            zorder=3)

    # Add state-id labels in the top-left of each cell using
    # display.add_label_to_plot.
    state_ids = np.arange(env.nS).reshape(env.shape)
    display.add_label_to_plot(
        ax, state_ids,
        shift=-0.35, size='6', env=env, skip_goals=False,
        format_fn=lambda v: str(int(v)),
    )

    cycle_descs = []
    for ci, cycle in enumerate(result['cycles']):
        basin_sz = result['basin_sizes'][ci]
        cycle_str = ', '.join(str(int(s)) for s in cycle)
        cycle_descs.append(f'{{{cycle_str}}} ({basin_sz})')
    ax.set_xlabel('  '.join(cycle_descs), fontsize=7)

    ax.set_title(f'label {ACTION_LABELS[a]} - {n_basins} cycle(s)', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim(-0.5, w_grid - 0.5)
    ax.set_ylim(h_grid - 0.5, -0.5)

fig_decomp.suptitle(r'Functional graph decomposition: cycles and basins under repeated single-label dynamics',
                    fontsize=11, y=1.02)
fig_decomp.tight_layout()

save_figure_variants(fig_decomp, FIGURES_DIR / 'figure-6-functional-graph-decomposition')
plt.show()

# Print summary for the record
print()
for a in range(env.nA):
    succ = _deterministic_successor(env, a)
    result = _find_basins(succ)
    print(f'Label {ACTION_LABELS[a]}: {len(result["cycles"])} cycle(s)')
    for ci, cycle in enumerate(result['cycles']):
        cstates = [int(s) for s in cycle]
        bsn = result['basin_sizes'][ci]
        print(f'  cycle {ci}: {cstates}  basin size: {bsn}')


## Panel B: Dominant action label for all 49 goals

7×7 grid of panels, each positioned at the goal's grid location.
Each panel shows $\arg\max_a \pi^*(a|s)$ under the twisted policy at $\beta=1$.

In [ ]:
# load all 49 policies from hive cache
all_policies = {}
all_infos = {}
for g in range(nS):
    data = np.load(HIVE_DIR / f'goal-{g}.npz')
    all_policies[g] = data['policy']
    all_infos[g] = data['info']

print(f'Loaded policies and info for {len(all_policies)} goals from {HIVE_DIR.name}')

In [ ]:
# Habit-cycle states from the functional graph decomposition.
# These are the exact cycle states of the deterministic per-label dynamics
# (det = 1 skeleton), matching `generate_run_comparison.py` and Panel A2 above.
# Replaces an earlier stationary-threshold heuristic that could miss or
# over-pick cells when cycle mass and basin-transient mass overlap.

def _det_succ(env, a):
    T = np.asarray(env.T)
    return np.array([int(np.argmax(T[s * env.nA + a])) for s in range(env.nS)])


def _cycle_states(env, a):
    succ = _det_succ(env, a)
    visited = np.full(env.nS, -1, dtype=int)
    cycle_set = set()
    for start in range(env.nS):
        if visited[start] >= 0:
            continue
        path = []
        s = start
        while visited[s] < 0:
            visited[s] = start
            path.append(s)
            s = int(succ[s])
        if visited[s] == start:
            cycle_set.update(path[path.index(s):])
    return sorted(int(s) for s in cycle_set)


attractors = {a: _cycle_states(env, a) for a in range(env.nA)}
for a in range(env.nA):
    print(f'{ACTION_LABELS[a]}: cycle states={attractors[a]}  '
          f'mass={sum(stationary[a][s] for s in attractors[a]):.2f}')

def _dominant_action_panel(ax, env, goal, policy):
    """Dominant action with faded background + attractor labels + attractor panel border."""
    plot_dominant_action_panel(
        ax, env, policy,
        goal=goal, title=None,
        goal_marker='x',
        goal_marker_kw=dict(color='black', markersize=5, markeredgewidth=1.5),
        alpha=0.45,
        attractor_states=attractors,
    )
    # colour panel border if this goal is an attractor state
    apply_attractor_panel_border(ax, goal, attractors, env=env)

fig_dom = plot_goal_grid(
    env, all_policies, _dominant_action_panel,
    cell_size=0.95,
    gap=0.08,
    title='Dominant action per state for goal state noted by x',
    footer_text=FOOTER,
)

# single centred legend line
from gridvis.display_twist import _action_color_indices, _select_action_colours
color_ids, palette_size = _action_color_indices(ACTION_LABELS)
action_colors = _select_action_colours(palette_size)

fs = 8
y_leg = 0.035
x = 0.07

fig_dom.text(x, y_leg, 'dominant action:  ', fontsize=fs, color='#333333', va='center', ha='left')
x += 0.145
for a in range(env.nA):
    c = action_colors[color_ids[a]]
    fig_dom.text(x, y_leg, '\u25a0 ' + ACTION_LABELS[a], fontsize=fs, color=c, va='center', ha='left')
    x += 0.055
x += 0.04
fig_dom.text(x, y_leg, 'attractor for label indicated by letters on graph:  ', fontsize=fs-1,
             color='#333333', va='center', ha='left')
x += 0.36
for a in range(env.nA):
    c = action_colors[color_ids[a]]
    fig_dom.text(x, y_leg, ACTION_LABELS[a], fontsize=fs, color=c, va='center', ha='left')
    x += 0.03

save_figure_variants(fig_dom, FIGURES_DIR / 'figure-6-dominant-action-all-goals')
plt.show()

## Panel C: Decision information for all 49 goals

Same 7×7 grid layout. Each panel shows per-state $\mathcal{I}_{\!D}^{\pi^*_\beta}(s)$ with the
policy quiver overlaid.

In [ ]:
# DI clim: 0–5 with SOFT_ORANGES
DI_CLIM = (0.0, 5.0)

def _di_panel(ax, env, goal, info_vec):
    """Render one DI heatmap + quiver panel for plot_goal_grid (no labels)."""
    from gridvis.display_twist_analysis import _format_axes
    _format_axes(ax, env)
    heatmap = info_vec.reshape(env.shape)
    display.heatmap_imshow(env, heatmap, ax, cmap=display.SOFT_ORANGES,
                          vmin=DI_CLIM[0], vmax=DI_CLIM[1])
    display.plot_quiver(env, all_policies[goal], ax=ax, skip_walls=True)
    display.add_rectangle_to_grid_for_walls(ax, env)
    display.add_outline_for_cells(ax, env)
    if getattr(getattr(env, 'actions', None), 'wrap', False):
        apply_dashed_border(ax)
    apply_attractor_panel_border(ax, goal, attractors, env=env)
    gy, gx = divmod(goal, env.shape[1])
    ax.plot(gx + 0.5, gy + 0.5, 'x', color='black',
            markersize=5, markeredgewidth=1.5, zorder=5)

fig_di = plot_goal_grid(
    env, all_infos, _di_panel,
    cell_size=0.95,
    gap=0.08,
    title=r'Decision information per goal, goal state noted by x',
    footer_text=FOOTER,
    clim=DI_CLIM,
    clim_cmap=display.SOFT_ORANGES,
    clim_label=(r'$\mathcal{I}_{\!D}^{\pi^*_\beta}(s)$'
                r'  where  '
                r'$\pi^*_\beta = \operatorname{arg\,min}_\pi\, \mathcal{F}^{\pi}_{\beta}(s)$'
                r',  $\beta=1$'),
)
save_figure_variants(fig_di, FIGURES_DIR / 'figure-6-di-all-goals')
plt.show()

## Export panels for LaTeX composition

Save (a), (b), (c) as separate PDFs. The combined figure is assembled in the
paper's LaTeX source using subfigure/minipage for consistent fonts and spacing.

In [ ]:
# save three panels as separate files for LaTeX composition
# (a) physical direction row — no colorbar since we have d^a labels
fig_a, _ = plot_physical_direction_row(
    env, stationary,
    show_attractors=False, show_action_legend=True,
    label=True, show_colorbar=False,
    footer_text=None, title=None,
)
save_figure_variants(fig_a, FIGURES_DIR / 'figure-6a-physical-direction')
plt.close(fig_a)
print('Saved figure-6a')

# (b) dominant action grid — no legend, sigma/run as two-line footer
#     (matches the DI grid height which has colorbar + footer)
fig_b = plot_goal_grid(
    env, all_policies, _dominant_action_panel,
    cell_size=0.95, gap=0.08, title=None,
    footer_text=f"$\\sigma$ = {SIGMA_HASH}\nrun = {RUN_NAME}",
)
save_figure_variants(fig_b, FIGURES_DIR / 'figure-6b-dominant-action')
plt.close(fig_b)
print('Saved figure-6b')

# (c) DI grid
fig_c = plot_goal_grid(
    env, all_infos, _di_panel,
    cell_size=0.95, gap=0.08, title=None, footer_text=None,
    clim=DI_CLIM, clim_cmap=display.SOFT_ORANGES,
    clim_label=(r'$\mathcal{I}_{\!D}^{\pi^*_\beta}(s)$'
                r'  where  '
                r'$\pi^*_\beta = \operatorname{arg\,min}_\pi\, \mathcal{F}^{\pi}_{\beta}(s)$'
                r',  $\beta=1$'),
)
save_figure_variants(fig_c, FIGURES_DIR / 'figure-6c-di')
plt.close(fig_c)
print('Saved figure-6c')

print(f'\nAll panels saved to {FIGURES_DIR}')
print('Compose in LaTeX with the snippet in the paper source.')

## Alternative: Compact Figure 6

Top row: 4 "following label" panels (physical direction + d^a labels).
Bottom row: 2 selected goals, each showing dominant action (left) and DI (right).

This version fits within the page limit while still showing the key structure.

In [ ]:
SELECTED_GOALS = [24, 48]

from gridvis.display_twist_analysis import (
    plot_physical_direction_panel, plot_dominant_action_panel,
    apply_dashed_border, apply_attractor_panel_border,
    _format_axes,
)
from matplotlib.patches import Rectangle as _Rect

# neutral cmap for occupancy heatmap
OCCUPANCY_CMAP = display.make_soft_cmap('Greys', minval=0.10, maxval=0.65)

# DI colour scale: 0–4 with SOFT_ORANGES
DI_CLIM_ALT = (0.0, 6.0)

# consistent title fontsize
TITLE_FS = 9

def _add_goal_outline(ax, env, color='gold', lw=2.0):
    for g in env.goals:
        gy, gx = divmod(int(g), env.shape[1])
        ax.add_patch(_Rect((gx + 0.05, gy + 0.05), 0.9, 0.9,
                           facecolor='none', edgecolor=color,
                           linewidth=lw, zorder=8))

# tight sizing
panel_w = 2.15
panel_h = panel_w
gap = 0.10
row_gap = 0.35
legend_h = 0.22
footer_h = 0.15
pad_t = 0.08
pad_b = footer_h + 0.02

fig_w = 4 * panel_w + 3 * gap + 0.15
fig_h = pad_b + legend_h + panel_h + row_gap + panel_h + pad_t

fig_alt = plt.figure(figsize=(fig_w, fig_h))

pw = panel_w / fig_w
ph = panel_h / fig_h

# --- top row: 4 following-label panels ---
y_top = (pad_b + legend_h + panel_h + row_gap) / fig_h
for i, a in enumerate(range(env.nA)):
    x = (0.075 + i * (panel_w + gap)) / fig_w
    ax = fig_alt.add_axes([x, y_top, pw, ph])
    plot_physical_direction_panel(
        ax, env, a,
        heatmap_var=stationary[a],
        cmap=OCCUPANCY_CMAP,
        show_attractors=False,
        label=True, label_size="6",
    )
    ax.set_title(f'following label {ACTION_LABELS[a]}', fontsize=TITLE_FS)

# --- bottom row: goal 1 (dom, di), goal 2 (dom, di) ---
y_bot = (pad_b + legend_h) / fig_h

for gi, goal in enumerate(SELECTED_GOALS):
    env_goal = build_twisted_env(goal=goal)

    x_dom = (0.075 + gi * 2 * (panel_w + gap)) / fig_w
    ax_dom = fig_alt.add_axes([x_dom, y_bot, pw, ph])
    plot_dominant_action_panel(
        ax_dom, env_goal, all_policies[goal],
        goal=None, title=None,
        alpha=0.45,
        attractor_states=attractors,
    )
    _add_goal_outline(ax_dom, env_goal)
    apply_attractor_panel_border(ax_dom, goal, attractors, env=env_goal)
    ax_dom.set_title(f'goal {goal}: dominant action', fontsize=TITLE_FS)

    x_di = (0.075 + (gi * 2 + 1) * (panel_w + gap)) / fig_w
    ax_di = fig_alt.add_axes([x_di, y_bot, pw, ph])
    _format_axes(ax_di, env_goal)
    heatmap = all_infos[goal].reshape(env_goal.shape)
    display.heatmap_imshow(env_goal, heatmap, ax_di, cmap=display.SOFT_ORANGES,
                           vmin=DI_CLIM_ALT[0], vmax=DI_CLIM_ALT[1])
    display.add_label_to_plot(ax_di, heatmap, shift=0.35, size='6', env=env_goal,
                              format_fn=display.format_number_2dp)
    display.plot_quiver(env_goal, all_policies[goal], ax=ax_di, skip_walls=True)
    _add_goal_outline(ax_di, env_goal)
    display.add_rectangle_to_grid_for_walls(ax_di, env_goal)
    display.add_outline_for_cells(ax_di, env_goal)
    if getattr(getattr(env_goal, 'actions', None), 'wrap', False):
        apply_dashed_border(ax_di)
    apply_attractor_panel_border(ax_di, goal, attractors, env=env_goal)
    DI_LABEL = (r'goal ' + str(goal) + r': $\mathcal{I}_{\!D}^{\pi^*}(s)$')
    ax_di.set_title(DI_LABEL, fontsize=TITLE_FS)

# legend — compact single line
cids_l, ps_l = _action_color_indices(ACTION_LABELS)
colors_l = _select_action_colours(ps_l)
fs = 6.5
y_leg = (pad_b + legend_h * 0.45) / fig_h
x_l = 0.04
fig_alt.text(x_l, y_leg, 'dominant action:', fontsize=fs, color='#333333', va='center', ha='left')
x_l += 0.09
for a in range(env.nA):
    c = colors_l[cids_l[a]]
    fig_alt.text(x_l, y_leg, '\u25a0 ' + ACTION_LABELS[a], fontsize=fs, color=c, va='center', ha='left')
    x_l += 0.035
x_l += 0.015
fig_alt.text(x_l, y_leg, 'attractor:', fontsize=fs, color='#333333', va='center', ha='left')
x_l += 0.055
for a in range(env.nA):
    c = colors_l[cids_l[a]]
    fig_alt.text(x_l, y_leg, ACTION_LABELS[a], fontsize=fs, color=c, va='center', ha='left')
    x_l += 0.018
x_l += 0.015
fig_alt.text(x_l, y_leg, 'goal: gold outline', fontsize=fs, color='#333333', va='center', ha='left')

fig_alt.text(0.99, 0.002, FOOTER, ha='right', va='bottom', fontsize=5, color='#aaaaaa')

save_figure_variants(fig_alt, FIGURES_DIR / 'figure-6-alt-compact')
plt.show()
print(f'Saved figure-6-alt-compact  (goals {SELECTED_GOALS})')

## Alternative goal pairing with hatched attractors

Try different goal pairs. Attractor cells are hatched in the colour of their
label rather than just marked with letters, making them more visible.

In [ ]:
ALT_GOALS = [24, 48]  # try different pairings here: [17, 33], [24, 0], [3, 40], etc.

from matplotlib.patches import Rectangle as _Rect
from gridvis.display_twist import _action_color_indices, _select_action_colours
import matplotlib.colors as mcolors

# action colours for hatching
cids_h, ps_h = _action_color_indices(ACTION_LABELS)
action_colors_h = _select_action_colours(ps_h)

def _add_attractor_hatching(ax, env, attractors, action_colors, color_ids):
    """Hatch attractor cells in the colour of their label."""
    h, w = env.shape
    for a, states in attractors.items():
        c = action_colors[color_ids[a]]
        for s in states:
            sy, sx = divmod(int(s), w)
            ax.add_patch(_Rect(
                (sx + 0.05, sy + 0.05), 0.9, 0.9,
                facecolor='none', edgecolor=c,
                linewidth=0.0, hatch='///',
                zorder=7, alpha=0.7,
            ))

# --- figure layout (same as compact figure) ---
panel_w = 2.15
panel_h = panel_w
gap = 0.10
row_gap = 0.35
legend_h = 0
footer_h = 0.15
pad_t = 0.08
pad_b = footer_h + 0.02

fig_w = 4 * panel_w + 3 * gap + 0.15
fig_h = pad_b + legend_h + panel_h + row_gap + panel_h + pad_t

fig_v2 = plt.figure(figsize=(fig_w, fig_h))

pw = panel_w / fig_w
ph = panel_h / fig_h

# --- top row: 4 following-label panels with hatched attractors ---
y_top = (pad_b + legend_h + panel_h + row_gap) / fig_h
for i, a in enumerate(range(env.nA)):
    x = (0.075 + i * (panel_w + gap)) / fig_w
    ax = fig_v2.add_axes([x, y_top, pw, ph])
    plot_physical_direction_panel(
        ax, env, a,
        heatmap_var=stationary[a],
        cmap=OCCUPANCY_CMAP,
        show_attractors=False,
        label=True, label_size="6",
    )
    _add_attractor_hatching(ax, env, {a: attractors[a]}, action_colors_h, cids_h)
    ax.set_title(f'following label {ACTION_LABELS[a]}', fontsize=TITLE_FS)

# --- bottom row: goal pairs (dom + di) with hatched attractors ---
y_bot = (pad_b + legend_h) / fig_h

for gi, goal in enumerate(ALT_GOALS):
    env_goal = build_twisted_env(goal=goal)

    x_dom = (0.075 + gi * 2 * (panel_w + gap)) / fig_w
    ax_dom = fig_v2.add_axes([x_dom, y_bot, pw, ph])
    plot_dominant_action_panel(
        ax_dom, env_goal, all_policies[goal],
        goal=None, title=None,
        alpha=0.45,
        attractor_states=attractors,
    )
    _add_attractor_hatching(ax_dom, env_goal, attractors, action_colors_h, cids_h)
    _add_goal_outline(ax_dom, env_goal)
    apply_attractor_panel_border(ax_dom, goal, attractors, env=env_goal)
    ax_dom.set_title(f'goal {goal}: dominant action', fontsize=TITLE_FS)

    x_di = (0.075 + (gi * 2 + 1) * (panel_w + gap)) / fig_w
    ax_di = fig_v2.add_axes([x_di, y_bot, pw, ph])
    _format_axes(ax_di, env_goal)
    heatmap = all_infos[goal].reshape(env_goal.shape)
    display.heatmap_imshow(env_goal, heatmap, ax_di, cmap=display.SOFT_ORANGES,
                           vmin=DI_CLIM_ALT[0], vmax=DI_CLIM_ALT[1])
    display.add_label_to_plot(ax_di, heatmap, shift=0.35, size='6', env=env_goal,
                              format_fn=display.format_number_2dp)
    display.plot_quiver(env_goal, all_policies[goal], ax=ax_di, skip_walls=True)
    _add_goal_outline(ax_di, env_goal)
    display.add_rectangle_to_grid_for_walls(ax_di, env_goal)
    display.add_outline_for_cells(ax_di, env_goal)
    if getattr(getattr(env_goal, 'actions', None), 'wrap', False):
        apply_dashed_border(ax_di)
    apply_attractor_panel_border(ax_di, goal, attractors, env=env_goal)
    DI_LABEL = (r'goal ' + str(goal) + r': $\mathcal{I}_{\!D}^{\pi^*}(s)$')
    ax_di.set_title(DI_LABEL, fontsize=TITLE_FS)

# legend
fs = 9
y_leg = (pad_b + legend_h * 0.45) / fig_h
y_leg = 0.002
x_l = 0.04
fig_v2.text(x_l, y_leg, 'dominant action:', fontsize=fs, color='#333333', va='center', ha='left')
x_l += 0.13
for a in range(env.nA):
    c = action_colors_h[cids_h[a]]
    fig_v2.text(x_l, y_leg, '\u25a0 ' + ACTION_LABELS[a], fontsize=fs, color=c, va='center', ha='left')
    x_l += 0.035
x_l += 0.035
fig_v2.text(x_l, y_leg, 'attractor (hatched):', fontsize=fs, color='#333333', va='center', ha='left')
x_l += 0.155
for a in range(env.nA):
    c = action_colors_h[cids_h[a]]
    fig_v2.text(x_l, y_leg, ACTION_LABELS[a], fontsize=fs, color=c, va='center', ha='left')
    x_l += 0.018
x_l += 0.015
fig_v2.text(x_l, y_leg, 'goal: yellow outline', fontsize=fs, color='#333333', va='center', ha='left')

"""
sigma footer
fig_v2.text(0.99, 0.002, FOOTER, ha='right', va='bottom', fontsize=5, color='#aaaaaa')
"""
save_figure_variants(fig_v2, FIGURES_DIR / f'figure-6-alt-v2-goals-{ALT_GOALS[0]}-{ALT_GOALS[1]}')
plt.show()
print(f'Saved figure-6-alt-v2 (goals {ALT_GOALS})')

In [ ]:
# --- individual subplot exports for the ALIFE presentation ---
# Standalone renders of the composite's subplots (no cropping needed):
# per-goal (dominant action + DI) pairs, each following-label panel,
# and the four-label strip.

# Fresh unconditioned env: the arrows must show the physical effect of
# taking the label at each state, independent of any goal built earlier.
env_labels = build_twisted_env_no_goal()

def _export_label_panel(a):
    fw, fh = panel_w + 0.15, panel_h + 0.35
    fig = plt.figure(figsize=(fw, fh))
    ax = fig.add_axes([0.04, 0.01, panel_w / fw, panel_h / fh])
    plot_physical_direction_panel(
        ax, env_labels, a, heatmap_var=stationary[a], cmap=OCCUPANCY_CMAP,
        show_attractors=False, label=True, label_size="6")
    _add_attractor_hatching(ax, env_labels, {a: attractors[a]}, action_colors_h, cids_h)
    ax.set_title(f'following label {ACTION_LABELS[a]}', fontsize=TITLE_FS)
    save_figure_variants(fig, FIGURES_DIR / f'figure-6-panel-label-{ACTION_LABELS[a]}')
    plt.close(fig)

for a in range(env.nA):
    _export_label_panel(a)

fw = 4 * panel_w + 3 * gap + 0.3
fh = panel_h + 0.35
fig_strip = plt.figure(figsize=(fw, fh))
for i, a in enumerate(range(env.nA)):
    x = (0.1 + i * (panel_w + gap)) / fw
    ax = fig_strip.add_axes([x, 0.01, panel_w / fw, panel_h / fh])
    plot_physical_direction_panel(
        ax, env_labels, a, heatmap_var=stationary[a], cmap=OCCUPANCY_CMAP,
        show_attractors=False, label=True, label_size="6")
    _add_attractor_hatching(ax, env_labels, {a: attractors[a]}, action_colors_h, cids_h)
    ax.set_title(f'following label {ACTION_LABELS[a]}', fontsize=TITLE_FS)
save_figure_variants(fig_strip, FIGURES_DIR / 'figure-6-panel-label-strip')
plt.close(fig_strip)

def _export_goal_pair(goal):
    fw = 2 * panel_w + gap + 0.3
    fh = panel_h + 0.35
    fig = plt.figure(figsize=(fw, fh))
    env_goal = build_twisted_env(goal=goal)

    ax_dom = fig.add_axes([0.1 / fw, 0.01, panel_w / fw, panel_h / fh])
    plot_dominant_action_panel(
        ax_dom, env_goal, all_policies[goal], goal=None, title=None,
        alpha=0.45, attractor_states=attractors)
    _add_attractor_hatching(ax_dom, env_goal, attractors, action_colors_h, cids_h)
    _add_goal_outline(ax_dom, env_goal)
    apply_attractor_panel_border(ax_dom, goal, attractors, env=env_goal)
    ax_dom.set_title(f'goal {goal}: dominant action', fontsize=TITLE_FS)

    ax_di = fig.add_axes([(0.1 + panel_w + gap) / fw, 0.01, panel_w / fw, panel_h / fh])
    _format_axes(ax_di, env_goal)
    heatmap = all_infos[goal].reshape(env_goal.shape)
    display.heatmap_imshow(env_goal, heatmap, ax_di, cmap=display.SOFT_ORANGES,
                           vmin=DI_CLIM_ALT[0], vmax=DI_CLIM_ALT[1])
    display.add_label_to_plot(ax_di, heatmap, shift=0.35, size='6', env=env_goal,
                              format_fn=display.format_number_2dp)
    display.plot_quiver(env_goal, all_policies[goal], ax=ax_di, skip_walls=True)
    _add_goal_outline(ax_di, env_goal)
    display.add_rectangle_to_grid_for_walls(ax_di, env_goal)
    display.add_outline_for_cells(ax_di, env_goal)
    if getattr(getattr(env_goal, 'actions', None), 'wrap', False):
        apply_dashed_border(ax_di)
    apply_attractor_panel_border(ax_di, goal, attractors, env=env_goal)
    ax_di.set_title(r'goal ' + str(goal) + r': $\mathcal{I}_{\!D}^{\pi^*}(s)$',
                    fontsize=TITLE_FS)
    save_figure_variants(fig, FIGURES_DIR / f'figure-6-goal-{goal}-pair')
    plt.close(fig)

for goal in ALT_GOALS:
    _export_goal_pair(goal)

print('exported: 4 label panels, label strip, goal pairs', ALT_GOALS)

## Baseline: Identity sigma (no twist)

Same compact layout but with the untwisted Cartesian labelling.
Per-goal data loaded from the baseline_goals cache.

In [ ]:
# --- Baseline (identity sigma) compact figure ---
import sys
# wrap_grid_helpers is co-located in this notebooks dir (already on sys.path)
from wrap_grid_helpers import solve_goal_untwisted

# Build the baseline (identity sigma) env with NO absorbing goal so the
# per-label stationary distributions in the top row are not contaminated
# by goal absorption. The bottom row per-goal panels still use a
# goal-conditioned env (built per goal below).
env_base = build_env(cfg['env_id'], shape=tuple(cfg['shape']),
                     goal=0, determinism=cfg['determinism'])
# Override goals to make nothing absorbing
env_base.update_dynamics_for_goals([])
print(f'Baseline env: nS={env_base.nS}  goals={list(env_base.goals)}  '
      f'wrap={getattr(env_base.actions, "wrap", False)}')

# stationary distributions for identity sigma (uniform on torus, since the
# Cartesian labelling has no recurrent concentration)
stationary_base = {}
goal_tag_base = 'nogoal' if not list(env_base.goals) else 'goal' + '_'.join(str(int(g)) for g in sorted(env_base.goals))
stat_base_cache = CACHE_DIR / f'stationary-identity-{goal_tag_base}.npz'
if stat_base_cache.exists():
    cached = np.load(stat_base_cache)
    stationary_base = {a: cached[f'action_{a}'] for a in range(env_base.nA)}
    print(f'Loaded baseline stationary from cache: {stat_base_cache.name}')
else:
    for a in range(env_base.nA):
        pi = np.zeros((env_base.nS, env_base.nA)); pi[:, a] = 1.0
        stationary_base[a] = get_stationary_distribution_eigen_decomposition(env_base, pi)
    np.savez_compressed(stat_base_cache,
                        **{f'action_{a}': stationary_base[a] for a in range(env_base.nA)})
    print(f'Computed and cached baseline stationary: {stat_base_cache.name}')

# load per-goal baseline data (from baseline_goals cache)
base_policies = {}
base_infos = {}
for goal in SELECTED_GOALS:
    res = solve_goal_untwisted(
        {'env_id': cfg['env_id'], 'shape': list(cfg['shape']),
         'determinism': cfg['determinism']},
        goal=goal, beta=1.0,
    )
    base_policies[goal] = res['policy']
    base_infos[goal] = res['information']
print(f'Loaded baseline policies for goals {SELECTED_GOALS}')

# no attractors for identity sigma (uniform distribution)
attractors_base = {a: [] for a in range(env_base.nA)}

FOOTER_BASE = r'$\sigma$ = identity (no twist)'

# --- build compact figure ---
fig_base = plt.figure(figsize=(fig_w, fig_h))

# top row: following-label panels (identity = arrows point in label direction)
y_top = (pad_b + legend_h + panel_h + row_gap) / fig_h
for i, a in enumerate(range(env_base.nA)):
    x = (0.075 + i * (panel_w + gap)) / fig_w
    ax = fig_base.add_axes([x, y_top, pw, ph])
    plot_physical_direction_panel(
        ax, env_base, a,
        heatmap_var=stationary_base[a],
        cmap=OCCUPANCY_CMAP,
        show_attractors=False,
        label=True, label_size="6",
    )
    ax.set_title(f'following label {ACTION_LABELS[a]}', fontsize=TITLE_FS)

# bottom row: same selected goals, untwisted (per-goal envs are
# goal-conditioned, which is correct for per-goal policy plots)
y_bot = (pad_b + legend_h) / fig_h

for gi, goal in enumerate(SELECTED_GOALS):
    env_base_goal = build_env(cfg['env_id'], shape=tuple(cfg['shape']),
                              goal=goal, determinism=cfg['determinism'])

    # dominant action
    x_dom = (0.075 + gi * 2 * (panel_w + gap)) / fig_w
    ax_dom = fig_base.add_axes([x_dom, y_bot, pw, ph])
    plot_dominant_action_panel(
        ax_dom, env_base_goal, base_policies[goal],
        goal=None, title=None,
        alpha=0.45,
        attractor_states=attractors_base,
    )
    _add_goal_outline(ax_dom, env_base_goal)
    ax_dom.set_title(f'goal {goal}: dominant action', fontsize=TITLE_FS)

    # DI
    x_di = (0.075 + (gi * 2 + 1) * (panel_w + gap)) / fig_w
    ax_di = fig_base.add_axes([x_di, y_bot, pw, ph])
    _format_axes(ax_di, env_base_goal)
    heatmap = base_infos[goal].reshape(env_base_goal.shape)
    display.heatmap_imshow(env_base_goal, heatmap, ax_di, cmap=display.SOFT_ORANGES,
                           vmin=DI_CLIM_ALT[0], vmax=DI_CLIM_ALT[1])
    display.add_label_to_plot(ax_di, heatmap, shift=0.35, size='6', env=env_base_goal,
                              format_fn=display.format_number_2dp)
    display.plot_quiver(env_base_goal, base_policies[goal], ax=ax_di, skip_walls=True)
    _add_goal_outline(ax_di, env_base_goal)
    display.add_rectangle_to_grid_for_walls(ax_di, env_base_goal)
    display.add_outline_for_cells(ax_di, env_base_goal)
    if getattr(getattr(env_base_goal, 'actions', None), 'wrap', False):
        apply_dashed_border(ax_di)
    DI_LABEL = r'goal ' + str(goal) + r': $\mathcal{I}_{\!D}^{\pi^*}(s)$'
    ax_di.set_title(DI_LABEL, fontsize=TITLE_FS)

"""
# legend
y_leg = (pad_b + legend_h * 0.45) / fig_h
x_l = 0.04
fig_base.text(x_l, y_leg, 'dominant action:', fontsize=fs, color='#333333', va='center', ha='left')
x_l += 0.09
for a in range(env_base.nA):
    c = colors_l[cids_l[a]]
    fig_base.text(x_l, y_leg, '\u25a0 ' + ACTION_LABELS[a], fontsize=fs, color=c, va='center', ha='left')
    x_l += 0.035
x_l += 0.015
fig_base.text(x_l, y_leg, 'goal: gold outline', fontsize=fs, color='#333333', va='center', ha='left')
"""
fig_base.text(0.99, 0.002, FOOTER_BASE, ha='right', va='bottom', fontsize=5, color='#aaaaaa')

save_figure_variants(fig_base, FIGURES_DIR / 'figure-6-baseline-compact')
plt.show()
print(f'Saved figure-6-baseline-compact  (goals {SELECTED_GOALS})')


## Baseline: all-goals dominant action and DI grids

Same 7×7 goal grid layout as the twisted version, but for the untwisted
Cartesian baseline. Per-goal data loaded from the baseline_goals cache.

In [ ]:
# Load all 49 baseline (untwisted) policies and DI from cache
from wrap_grid_helpers import solve_goal_untwisted, build_untwisted_env

baseline_cfg = {
    'env_id': cfg['env_id'],
    'shape': list(cfg['shape']),
    'determinism': cfg['determinism'],
}

all_base_policies = {}
all_base_infos = {}
for g in range(nS):
    res = solve_goal_untwisted(baseline_cfg, goal=g, beta=1.0)
    all_base_policies[g] = res['policy']
    all_base_infos[g] = res['information']

print(f'Loaded baseline policies and info for {len(all_base_policies)} goals')

# Build baseline env for plotting (no twist)
env_base = build_untwisted_env(baseline_cfg, goal=0)

# No attractors for the baseline (identity sigma gives straight-line trajectories)
attractors_base = {a: [] for a in range(env_base.nA)}

FOOTER_BASE = (
    r'baseline (identity $\sigma$)'
    f'    run = {cfg.get("run_name", "")}'
)

In [ ]:
# --- Baseline: dominant action all goals ---

def _baseline_dom_panel(ax, env_ref, goal, policy):
    """Dominant action panel for baseline (no attractors)."""
    plot_dominant_action_panel(
        ax, env_ref, policy,
        goal=goal, title=None,
        goal_marker='x',
        goal_marker_kw=dict(color='black', markersize=5, markeredgewidth=1.5),
        alpha=0.45,
        attractor_states=attractors_base,
    )

fig_base_dom = plot_goal_grid(
    env_base, all_base_policies, _baseline_dom_panel,
    cell_size=0.95,
    gap=0.08,
    title='Baseline: dominant action per state for goal state noted by x',
    footer_text=FOOTER_BASE,
)

# legend
from gridvis.display_twist import _action_color_indices, _select_action_colours
color_ids_b, palette_size_b = _action_color_indices(ACTION_LABELS)
action_colors_b = _select_action_colours(palette_size_b)

fs = 8
y_leg = 0.035
x_l = 0.07
fig_base_dom.text(x_l, y_leg, 'dominant action:  ', fontsize=fs, color='#333333', va='center', ha='left')
x_l += 0.145
for a in range(env_base.nA):
    c = action_colors_b[color_ids_b[a]]
    fig_base_dom.text(x_l, y_leg, '\u25a0 ' + ACTION_LABELS[a], fontsize=fs, color=c, va='center', ha='left')
    x_l += 0.055
x_l += 0.04
fig_base_dom.text(x_l, y_leg, 'goal: gold outline', fontsize=fs, color='#333333', va='center', ha='left')

save_figure_variants(fig_base_dom, FIGURES_DIR / 'figure-6-baseline-dominant-action-all-goals')
plt.show()

In [ ]:
# --- Baseline: DI all goals ---

def _baseline_di_panel(ax, env_ref, goal, info_vec):
    """DI heatmap + quiver for baseline."""
    from gridvis.display_twist_analysis import _format_axes
    _format_axes(ax, env_ref)
    heatmap = info_vec.reshape(env_ref.shape)
    display.heatmap_imshow(env_ref, heatmap, ax, cmap=display.SOFT_ORANGES,
                          vmin=DI_CLIM[0], vmax=DI_CLIM[1])
    display.plot_quiver(env_ref, all_base_policies[goal], ax=ax, skip_walls=True)
    display.add_rectangle_to_grid_for_walls(ax, env_ref)
    display.add_outline_for_cells(ax, env_ref)
    if getattr(getattr(env_ref, 'actions', None), 'wrap', False):
        apply_dashed_border(ax)
    gy, gx = divmod(goal, env_ref.shape[1])
    ax.plot(gx + 0.5, gy + 0.5, 'x', color='black',
            markersize=5, markeredgewidth=1.5, zorder=5)

fig_base_di = plot_goal_grid(
    env_base, all_base_infos, _baseline_di_panel,
    cell_size=0.95,
    gap=0.08,
    title=r'Baseline: decision information per goal, goal state noted by x',
    footer_text=FOOTER_BASE,
    clim=DI_CLIM,
    clim_cmap=display.SOFT_ORANGES,
    clim_label=(r'$\mathcal{I}_{\!D}^{\pi^*_\beta}(s)$'
                r'  where  '
                r'$\pi^*_\beta = \operatorname{arg\,min}_\pi\, \mathcal{F}^{\pi}_{\beta}(s)$'
                r',  $\beta=1$'),
)

save_figure_variants(fig_base_di, FIGURES_DIR / 'figure-6-baseline-di-all-goals')
plt.show()